# MORICHIKA Heavy RAG Gemma31B v1

Native strict-v3 retrieval, context-policy-v4 and Gemma31B dual-order verification. No legacy notebook/data-dump logic.

In [ ]:
from __future__ import annotations

import gc, hashlib, importlib, importlib.metadata, json, math, os, random, re
import shutil, sqlite3, stat, subprocess, sys, time, unicodedata, zipfile
from collections import Counter, defaultdict
from pathlib import Path, PurePosixPath
from typing import Any, Callable

import numpy as np
import pandas as pd

SEED = 20260720
RUN_LLM = True
MODEL_BACKEND = "q4_gguf"
MODEL_ID = "google/gemma-4-31B-it"
MODEL_PATH_OVERRIDE = ""
Q4_MODEL_ID = "google/gemma-4-31b-it-qat-q4_0-gguf"
Q4_MODEL_PATH_OVERRIDE = "/kaggle/input/models/google/gemma-4/gguf/gemma-4-31b-it-qat-q4_0-gguf/2/gemma-4-31B_q4_0-it.gguf"
Q4_N_CTX, Q4_N_BATCH, Q4_N_UBATCH = 2048, 384, 128
Q4_CONTEXT_CHAR_FALLBACKS = (6000, 3600, 2000, 1000, 400, 0)
MAX_LENGTH, BATCH_ROWS, CHECKPOINT_EVERY = 2048, 2, 25
MAX_REFERENCE_ANSWERS = 12
MAX_DELIB_TOKENS = 12
DELIB_BATCH_ROWS = 2
MAX_DELIB_SAMPLE_ROWS = MAX_DELIB_TEST_ROWS = 0
RUN_DELIBERATION = False
ALLOW_ONLINE_MODEL_FALLBACK = False
MIN_SEMANTIC_REFERENCE_SAMPLE_AGREEMENTS = 0

random.seed(SEED); np.random.seed(SEED)
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
INPUT_ROOT = Path("/kaggle/input")
WORK_DIR = Path("/kaggle/working")
WORK_DIR.mkdir(parents=True, exist_ok=True)
if not INPUT_ROOT.is_dir():
    raise RuntimeError("This production notebook must run inside Kaggle")

test_path = INPUT_ROOT / "competitions/bengali-hallucination/test set.csv"
sample_path = INPUT_ROOT / "competitions/bengali-hallucination/sample submission.csv"
if not test_path.is_file() or not sample_path.is_file():
    raise FileNotFoundError("Competition test/template files are not attached")
test = pd.read_csv(test_path)
sample_submission = pd.read_csv(sample_path)
required = {"id", "context", "prompt_bn", "response_bn"}
if not required.issubset(test.columns):
    raise ValueError(f"test schema missing: {sorted(required-set(test.columns))}")
for name in ("context", "prompt_bn", "response_bn"):
    test[name] = test[name].fillna("").astype(str)
test["id"] = test["id"].astype(str)
sample_submission["id"] = sample_submission["id"].astype(str)
if test.id.duplicated().any() or test.id.tolist() != sample_submission.id.tolist():
    raise ValueError("competition id/order contract failed")

NULL_CONTEXTS = {"", "none", "null", "nan", "n/a", "na", "[null]", "[none]", "[nan]", "[n/a]", "[na]", "<null>", "<none>", "<nan>"}
def has_context(value: object) -> bool:
    folded=str(value or "").strip().casefold()
    if folded in NULL_CONTEXTS or re.fullmatch(r"\[\s*(?:null|none|nan|n/a|na)?\s*\]",folded):
        return False
    return True
def clean_markup(value: object) -> str:
    return str(value or "").replace("\r\n", "\n").replace("\r", "\n").strip()
def row_references(row):
    return (), ()

context_count=int(test.context.map(has_context).sum())
if len(test)==2516 and context_count!=1361:
    raise RuntimeError(f"Phase1 lane canary failed: expected context=1361 closed=1155; got context={context_count} closed={len(test)-context_count}")
print("MORICHIKA test rows:", len(test), "context:", context_count, "closed:", len(test)-context_count)


In [ ]:
EXPECTED_DATASET_ID = "ishtyy/morichika-phase2-retrieval-strict-v3-20260720"
EXPECTED_MANIFEST_ID = "bfd6eecb3d011462d45282aa79967cd5258a90c6c5324193b5bb28d5a3c439b8"
EXPECTED_PACKAGE_ID = "6f68b6f284d55bacc3a34530a8492a78347b5c01243d55d256eb467ea096ad7e"

def file_sha256(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(8 * 1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def canonical_json(value: Any) -> str:
    return json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(",", ":"))

def validate_manifest(manifest: dict) -> None:
    core = {k: v for k, v in manifest.items() if k != "manifest_id"}
    if manifest.get("dataset_id") != EXPECTED_DATASET_ID:
        raise RuntimeError("wrong MORICHIKA retrieval dataset")
    if manifest.get("manifest_id") != EXPECTED_MANIFEST_ID or hashlib.sha256(canonical_json(core).encode()).hexdigest() != EXPECTED_MANIFEST_ID:
        raise RuntimeError("MORICHIKA manifest identity mismatch")
    if manifest.get("package_id") != EXPECTED_PACKAGE_ID:
        raise RuntimeError("MORICHIKA package identity mismatch")
    files = manifest.get("files")
    if not isinstance(files, list) or len(files) < 90:
        raise RuntimeError("MORICHIKA payload declaration incomplete")

def verify_tree(root: Path, manifest: dict) -> Path:
    for spec in manifest["files"]:
        rel = PurePosixPath(str(spec["path"]))
        if rel.is_absolute() or ".." in rel.parts:
            raise RuntimeError(f"unsafe declared path: {rel}")
        path = root.joinpath(*rel.parts)
        if not path.is_file() or path.is_symlink():
            raise FileNotFoundError(f"declared MORICHIKA file missing: {rel}")
        if path.stat().st_size != int(spec["bytes"]) or file_sha256(path) != str(spec["sha256"]):
            raise RuntimeError(f"MORICHIKA hash/size mismatch: {rel}")
    return root

def materialize_morichika() -> tuple[Path, dict]:
    direct = []
    for path in INPUT_ROOT.rglob("bundle_manifest.json"):
        try:
            value = json.loads(path.read_text(encoding="utf-8-sig"))
        except Exception:
            continue
        if isinstance(value, dict) and value.get("manifest_id") == EXPECTED_MANIFEST_ID:
            direct.append((path, value))
    if len(direct) == 1:
        path, manifest = direct[0]
        validate_manifest(manifest)
        return verify_tree(path.parent, manifest), manifest
    if len(direct) > 1:
        raise RuntimeError(f"ambiguous MORICHIKA direct bundles: {len(direct)}")

    matches = []
    for archive_path in INPUT_ROOT.rglob("payload.zip"):
        try:
            with zipfile.ZipFile(archive_path) as archive:
                infos = archive.infolist()
                if len({i.filename for i in infos}) != len(infos):
                    raise RuntimeError("duplicate payload.zip members")
                for info in infos:
                    p = PurePosixPath(info.filename)
                    mode = (info.external_attr >> 16) & 0xFFFF
                    if p.is_absolute() or ".." in p.parts or "\\" in info.filename or info.flag_bits & 1 or stat.S_ISLNK(mode):
                        raise RuntimeError(f"unsafe payload.zip member: {info.filename}")
                    if p.name == "bundle_manifest.json" and info.file_size < 2 * 1024 * 1024:
                        value = json.loads(archive.read(info).decode("utf-8-sig"))
                        if value.get("manifest_id") == EXPECTED_MANIFEST_ID:
                            matches.append((archive_path, info.filename, value))
        except zipfile.BadZipFile:
            continue
    if len(matches) != 1:
        raise RuntimeError(f"expected one hash-bound MORICHIKA payload.zip, found {len(matches)}")
    archive_path, manifest_name, manifest = matches[0]
    validate_manifest(manifest)
    prefix = PurePosixPath(manifest_name).parent
    declared = {str(s["path"]): s for s in manifest["files"]}
    mapped = {}
    with zipfile.ZipFile(archive_path) as archive:
        for info in archive.infolist():
            if info.is_dir():
                continue
            p = PurePosixPath(info.filename)
            if tuple(p.parts[:len(prefix.parts)]) != tuple(prefix.parts):
                raise RuntimeError(f"mixed payload prefix: {info.filename}")
            rel = PurePosixPath(*p.parts[len(prefix.parts):]).as_posix()
            if rel == "bundle_manifest.json":
                continue
            if rel == "dataset-metadata.json":
                metadata_value=json.loads(archive.read(info).decode("utf-8-sig"))
                if metadata_value.get("id") != EXPECTED_DATASET_ID:
                    raise RuntimeError("nested dataset metadata identity mismatch")
                continue
            if rel not in declared or rel in mapped:
                raise RuntimeError(f"undeclared/duplicate payload member: {rel}")
            mapped[rel] = info
        if set(mapped) != set(declared):
            raise RuntimeError("payload.zip is incomplete")
        stage = WORK_DIR / "morichika_strict_v3"
        if stage.exists(): shutil.rmtree(stage)
        stage.mkdir(parents=True)
        (stage / "bundle_manifest.json").write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
        for rel, info in mapped.items():
            spec = declared[rel]
            if info.file_size != int(spec["bytes"]):
                raise RuntimeError(f"zip member size mismatch: {rel}")
            dst = stage.joinpath(*PurePosixPath(rel).parts)
            dst.parent.mkdir(parents=True, exist_ok=True)
            h = hashlib.sha256(); written = 0
            with archive.open(info) as src, dst.open("wb") as out:
                while True:
                    chunk = src.read(8 * 1024 * 1024)
                    if not chunk: break
                    out.write(chunk); h.update(chunk); written += len(chunk)
            if written != int(spec["bytes"]) or h.hexdigest() != spec["sha256"]:
                raise RuntimeError(f"zip member hash mismatch: {rel}")
    return verify_tree(stage, manifest), manifest

MORICHIKA_ROOT, MORICHIKA_MANIFEST = materialize_morichika()
sys.path.insert(0, str(MORICHIKA_ROOT))
print("MORICHIKA corpus root:", MORICHIKA_ROOT)
print("MORICHIKA package:", EXPECTED_PACKAGE_ID, "files:", len(MORICHIKA_MANIFEST["files"]))


In [ ]:
"""Context-only policy packet for the MORICHIKA generalized hybrid.

The module is intentionally self-contained so its exact source can be embedded
in an offline Kaggle notebook.  It never retrieves, labels, or consults a
competition row identifier.  It converts the supplied context, question and
candidate response into an auditable instruction packet for the verifier.
"""

from __future__ import annotations

import hashlib
import json
import re
import unicodedata
from typing import Any, Callable


VERSION = "morichika-context-policy-v4-full-coverage-map-aggregate"

# These are reasoning checks, not predicted classes.  Keeping the recovered
# inventory explicit prevents a later prompt edit from silently dropping an
# edge-case family.
CANONICAL_POLICY_FAMILIES = (
    "question_grounding_answerability_and_premise_validity",
    "exact_entity_relation_property_and_answer_type",
    "direct_support_contradiction_and_partial_containment",
    "supplied_definition_formula_theory_and_rule_application",
    "date_event_role_and_as_of_time",
    "year_age_duration_calendar_and_timezone",
    "bounded_arithmetic_units_ratio_percentage_and_order",
    "kinship_and_relational_composition",
    "creator_founder_user_operator_office_title_and_jurisdiction",
    "birthplace_residence_nationality_and_event_participation",
    "legal_definition_section_effective_date_minimum_maximum_and_frequency",
    "numeric_whole_component_range_extremum_ordinal_and_granularity",
    "negation_quantifier_comparator_modality_and_clause_scope",
    "antonym_idiom_prefix_register_etymology_and_guruchandali",
    "samas_sandhi_affix_spelling_natva_and_satva",
    "unicode_joiner_conjunct_digit_punctuation_ocr_and_word_break",
    "ambiguity_conflict_invalid_premise_and_no_world_rescue",
)

ENGINEERED_EVALUATION_CELLS = (
    "context_only_lane", "full_context_boundary", "answerable_supported",
    "answerable_refuted", "genuinely_missing_nei", "same_passage_different_question",
    "entity_swap", "relation_swap", "answer_type_swap", "creator_vs_user_operator",
    "birthplace_vs_residence_event_place", "partial_answer_extra_claim",
    "theory_application", "formula_operand_application", "age_vs_year",
    "relative_timeline", "event_phase_order", "kinship_composition",
    "unit_or_number_swap", "negation_scope", "quantifier_comparator_scope",
    "grammar_operation_swap", "lexical_exact_vs_semantic_near",
    "unicode_equivalent", "unicode_word_break_non_equivalent", "cross_window_conflict",
)

OPERATION_AXIS = (
    "antonym_lookup", "birth_date_slot", "entity_text_relation",
    "event_date_or_status_slot", "idiom_meaning_lookup",
    "legal_definition_or_threshold", "legal_effective_date", "legal_maximum_fine",
    "legal_maximum_imprisonment", "legal_minimum_meeting_frequency",
    "location_slot", "numeric_or_ordinal_slot", "prefix_origin_classification",
    "samas_taxonomy", "sandhi_formation",
)

_SIGNALS: tuple[tuple[str, str], ...] = (
    ("year_age_duration", r"ব[য়য়]স|জন্ম|বিবাহ|বিয়ে|বিয়ে|বছর|সাল|year|age"),
    ("calendar_date", r"তারিখ|কবে|দিন|মাস|জানুয়ারি|ফেব্রুয়ারি|মার্চ|ডিসেম্বর"),
    ("timeline_event_order", r"আগে|পরে|তারপর|পরবর্তীতে|প্রথমে|শেষে|ঘটে|ঘটেছিল"),
    ("relative_time_offset", r"(?:বছর|দিন|মাস)\s*(?:আগে|পরে)|after|before|later"),
    ("kinship_relation_composition", r"বাবা|পিতা|মা|মাতা|দাদা|পিতামহ|নানা|grandfather"),
    ("bounded_arithmetic", r"কত|যোগ|বিয়োগ|বিয়োগ|গুণ|ভাগ|শতাংশ|হিসাব|calculate"),
    ("unit_dimension", r"কিলোমিটার|মিটার|গ্রাম|লিটার|সেকেন্ড|মিনিট|ঘণ্টা|টাকা|শতাংশ"),
    ("negation_scope", r"(?:^|\s)(?:না|নয়|নয়|নেই|নি)(?:\s|$)|হয়নি|হয়নি|করেনি"),
    ("quantifier_scope", r"সব|সকল|কোনো|কোনও|কেবল|শুধু|প্রত্যেক|একটিও|কিছু"),
    ("comparator_scope", r"সর্বাধিক|সর্বনিম্ন|বেশি|কম|পূর্ববর্তী|পরবর্তী|minimum|maximum"),
    ("modality_clause_scope", r"পারে|উচিত|অবশ্যই|সম্ভব|শর্তে|যদি|হলে"),
    ("definition_theory_rule_application", r"সংজ্ঞা|তত্ত্ব|সূত্র|নিয়ম|নিয়ম|অর্থ"),
    ("antonym_exact_pair", r"বিপরীত\s*শব্দ|antonym"),
    ("idiom_exact_meaning", r"বাগধারা|প্রবাদ"),
    ("prefix_suffix_origin", r"উপসর্গ|প্রত্যয়|প্রত্যয়"),
    ("samas_exact_operation", r"সমাস"),
    ("sandhi_exact_operands", r"সন্ধি"),
    ("register_etymology_guruchandali", r"গুরুচণ্ডালী|তৎসম|তদ্ভব|ফারসি|সংস্কৃত"),
    ("natva_satva_spelling", r"ণত্ব|ষত্ব|বানান"),
    ("creator_user_operator_role", r"স্রষ্টা|প্রতিষ্ঠাতা|নির্মাতা|ব্যবহারকারী|পরিচালক"),
    ("birthplace_residence_event_place", r"জন্মস্থান|বাসস্থান|রাজধানী|কোথায়|কোথায়|স্থান"),
    ("legal_definition_section", r"আইন|ধারা|বিধি|সংজ্ঞা"),
    ("minimum_maximum_frequency", r"সর্বনিম্ন|সর্বোচ্চ|কমপক্ষে|বেশিরভাগ|বার"),
    ("ordinal_list_order", r"প্রথম|দ্বিতীয়|দ্বিতীয়|তৃতীয়|তৃতীয়|ক্রম|তম"),
)

_BN_DIGITS = str.maketrans("০১২৩৪৫৬৭৮৯", "0123456789")
_JOINERS = {"\u200c", "\u200d", "\ufeff"}
_FIXED_LEXICAL_SHELLS = {
    "antonym_lookup": re.compile(r"(?:বিপরীত\s*শব্দ|বিপরীতার্থক\s*শব্দ).*(?:কী|কি|কোন)", re.IGNORECASE),
    "idiom_meaning_lookup": re.compile(r"(?:বাগধারা|প্রবাদ).*(?:অর্থ|মানে).*(?:কী|কি|কোন)", re.IGNORECASE),
    "prefix_origin_classification": re.compile(r"উপসর্গ.*(?:উৎস|শ্রেণি|জাতীয়|জাতীয়|কোন)", re.IGNORECASE),
}


def _sha(value: str) -> str:
    return hashlib.sha256(value.encode("utf-8")).hexdigest()


def _canonical(value: Any) -> str:
    return json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(",", ":"))


def comparison_view(value: str) -> str:
    """Bounded comparison only; never rewrites literal evidence."""
    return " ".join(unicodedata.normalize("NFC", value).translate(_BN_DIGITS).casefold().split())


def unicode_receipt(value: str) -> dict[str, Any]:
    nfc = unicodedata.normalize("NFC", value)
    return {
        "literal_sha256": _sha(value),
        "utf8_bytes": len(value.encode("utf-8")),
        "codepoints": len(value),
        "nfc_identical": nfc == value,
        "nfc_sha256": _sha(nfc),
        "joiner_positions": [index for index, char in enumerate(value) if char in _JOINERS],
        "replacement_character_positions": [index for index, char in enumerate(value) if char == "\ufffd"],
        "comparison_view_sha256": _sha(comparison_view(value)),
    }


def contextual_exact_lexical_policy(
    question: str, response: str, records: list[dict[str, Any]] | None
) -> dict[str, Any]:
    """Expose only the frozen exact lexical-shell exception, nonterminally."""
    folded_question = comparison_view(question)
    folded_response = comparison_view(response)
    operation = next(
        (name for name, pattern in _FIXED_LEXICAL_SHELLS.items() if pattern.search(folded_question)),
        None,
    )
    base = {
        "lookup_mode": "not_applicable",
        "operation": operation,
        "key": None,
        "source_ids": [],
        "record_sha256": [],
        "exact_operation": operation is not None,
        "exact_key": False,
        "exact_sense_register": False,
        "conflict": False,
        "conflict_status": "none",
        "generic_retrieval_invoked": False,
        "terminal_label_authority": False,
        "model_nonterminal": True,
        "evidence": [],
    }
    if operation is None:
        base["lookup_mode"] = "forbidden_for_ordinary_context"
        return base

    matched: list[dict[str, Any]] = []
    conflicting: list[dict[str, Any]] = []
    for record in records or []:
        if record.get("operation") != operation:
            continue
        terms = [str(record.get("term_key") or "")] + [str(v) for v in record.get("display_terms", [])]
        exact_terms = []
        for term in terms:
            key = comparison_view(term)
            if key and re.search(rf"(?<![\u0980-\u09FFA-Za-z0-9]){re.escape(key)}(?![\u0980-\u09FFA-Za-z0-9])", folded_question):
                exact_terms.append(key)
        if not exact_terms:
            continue
        sense_values = [str(record.get(name) or "") for name in ("sense", "register", "etymological_class")]
        required_scope = [comparison_view(value) for value in sense_values if value]
        exact_scope = not required_scope or all(value in folded_question for value in required_scope)
        candidate = {**record, "_matched_key": sorted(exact_terms, key=lambda value: (-len(value), value))[0], "_exact_scope": exact_scope}
        if record.get("conflict_status") == "none" and exact_scope:
            matched.append(candidate)
        else:
            conflicting.append(candidate)

    if not matched and not conflicting:
        base["lookup_mode"] = "exact_shell_no_exact_key_nonterminal"
        return base
    all_candidates = matched + conflicting
    keys = {value["_matched_key"] for value in all_candidates}
    accepted_sets = {
        tuple(sorted(comparison_view(v) for v in value.get("accepted_answers", []) if str(v).strip()))
        for value in matched
    }
    conflict = bool(conflicting) or len(keys) != 1 or len(accepted_sets) > 1
    base.update({
        "lookup_mode": "exact_hash_bound_lexical_policy" if not conflict else "exact_lexical_conflict_nei_nonterminal",
        "key": sorted(keys)[0] if len(keys) == 1 else None,
        "source_ids": sorted({str(value.get("source_id") or "") for value in all_candidates if value.get("source_id")}),
        "record_sha256": sorted({_sha(_canonical({k: v for k, v in value.items() if not str(k).startswith("_")})) for value in all_candidates}),
        "exact_key": len(keys) == 1,
        "exact_sense_register": all(value.get("_exact_scope") is True for value in all_candidates),
        "conflict": conflict,
        "conflict_status": "conflict" if conflict else "none",
    })
    if not conflict:
        for value in matched[:3]:
            accepted = [comparison_view(v) for v in value.get("accepted_answers", [])]
            contrast = [comparison_view(v) for v in value.get("contrast_answers", [])]
            base["evidence"].append({
                "source_id": value.get("source_id"),
                "operation": operation,
                "term_key": value.get("term_key"),
                "accepted_answers": value.get("accepted_answers", []),
                "contrast_answers": value.get("contrast_answers", []),
                "response_relation": (
                    "support_candidate" if folded_response in accepted else
                    "counter_candidate" if folded_response in contrast else "neutral_candidate"
                ),
            })
    return base


def detected_policy_families(context: str, question: str, response: str) -> list[str]:
    joined = "\n".join((question, response, context))
    detected = {
        "lane_context_only", "full_context_evidence_boundary", "question_answerability",
        "question_premise_validity", "exact_answer_slot", "same_passage_different_question",
        "entity_identity", "relation_property_identity", "answer_type_identity",
        "direct_support", "direct_contradiction", "partial_containment_extra_claim",
        "ambiguity_conflict_nei", "unicode_nfc_joiner_conjunct_digit", "ocr_word_break_caution",
    }
    for family, pattern in _SIGNALS:
        if re.search(pattern, joined, re.IGNORECASE):
            detected.add(family)
    if re.search(r"[০-৯0-9]", joined):
        detected.update({"number_whole_component_range", "formula_operand_application"})
    order = [name for name, _ in _SIGNALS]
    order += [
        "lane_context_only", "full_context_evidence_boundary", "question_answerability",
        "question_premise_validity", "exact_answer_slot", "same_passage_different_question",
        "entity_identity", "relation_property_identity", "answer_type_identity",
        "direct_support", "direct_contradiction", "partial_containment_extra_claim",
        "ambiguity_conflict_nei", "unicode_nfc_joiner_conjunct_digit", "ocr_word_break_caution",
        "number_whole_component_range", "formula_operand_application",
    ]
    return [family for family in dict.fromkeys(order) if family in detected]


def full_coverage_windows(
    context: str, *, max_chars: int = 3000, overlap_chars: int = 320
) -> list[dict[str, Any]]:
    """Partition every context character into overlapping replayable windows."""
    if not context:
        raise ValueError("context must be non-empty")
    if max_chars < 256 or overlap_chars < 32 or overlap_chars >= max_chars:
        raise ValueError("invalid full-coverage window geometry")
    windows: list[dict[str, Any]] = []
    start = 0
    while start < len(context):
        hard_end = min(len(context), start + max_chars)
        end = hard_end
        if hard_end < len(context):
            boundaries = [context.rfind(mark, start + max_chars // 2, hard_end) for mark in ("\n", "।", ".", "?", "!")]
            boundary = max(boundaries)
            if boundary >= start:
                end = boundary + 1
        literal = context[start:end]
        windows.append({
            "window_index": len(windows),
            "context_char_start": start,
            "context_char_end": end,
            "literal_text": literal,
            "literal_span_sha256": _sha(literal),
        })
        if end == len(context):
            break
        start = end - overlap_chars
    covered = [False] * len(context)
    for index, window in enumerate(windows):
        start, end = window["context_char_start"], window["context_char_end"]
        literal = window["literal_text"]
        if context[start:end] != literal or _sha(literal) != window["literal_span_sha256"]:
            raise ValueError(f"full-coverage window {index} does not replay")
        for position in range(start, end):
            covered[position] = True
    if not all(covered):
        raise ValueError("full-coverage window ledger contains a character gap")
    return windows


def build_window_adjudication_user(
    window: dict[str, Any], question: str, response: str, total_windows: int
) -> str:
    return (
        "WINDOW-LOCAL CONTEXT PASS. Judge only evidence present in this literal supplied-context window. "
        "A local miss is not a contradiction: output not_enough_information unless this window directly "
        "supports or directly refutes the exact question slot. Do not use outside knowledge.\n\n"
        f"QUESTION (literal):\n{question}\n\nCANDIDATE RESPONSE (literal):\n{response}\n\n"
        f"WINDOW {window['window_index'] + 1}/{total_windows} "
        f"[chars={window['context_char_start']}:{window['context_char_end']}; "
        f"sha256={window['literal_span_sha256']}]:\n{window['literal_text']}"
    )


def _literal_question_excerpt(
    window: dict[str, Any], question: str, max_chars: int
) -> dict[str, Any]:
    """Select a literal, question-conditioned excerpt without rewriting it."""
    literal = str(window["literal_text"])
    if max_chars < 32:
        max_chars = 32
    q_tokens = {
        token.casefold() for token in re.findall(r"[\u0980-\u09FFA-Za-z0-9]+", question)
        if len(token) > 1
    }
    candidates = list(re.finditer(r"[^।.!?\n]+[।.!?]?", literal))
    if candidates:
        def score(match: re.Match[str]) -> tuple[int, int]:
            tokens = {
                token.casefold() for token in re.findall(r"[\u0980-\u09FFA-Za-z0-9]+", match.group())
            }
            return (len(tokens & q_tokens), -match.start())
        best = max(candidates, key=score)
        local_start = max(0, best.start() - max_chars // 4)
        local_end = min(len(literal), local_start + max_chars)
    else:
        local_start, local_end = 0, min(len(literal), max_chars)
    excerpt = literal[local_start:local_end]
    absolute_start = int(window["context_char_start"]) + local_start
    absolute_end = absolute_start + len(excerpt)
    return {
        "excerpt_char_start": absolute_start,
        "excerpt_char_end": absolute_end,
        "literal_excerpt": excerpt,
        "literal_excerpt_sha256": _sha(excerpt),
    }


def build_aggregation_user(
    question: str,
    response: str,
    window_results: list[dict[str, Any]],
    *,
    selected_notes: list[dict[str, Any]] | None = None,
    bounded_derivations: list[dict[str, Any]] | None = None,
    lexical_policy: dict[str, Any] | None = None,
) -> str:
    """Create the exact-question final adjudication over all window passes."""
    compact = [
        {
            "window_index": row["window_index"],
            "context_char_start": row["context_char_start"],
            "context_char_end": row["context_char_end"],
            "literal_span_sha256": row["literal_span_sha256"],
            "window_verdict": row["window_verdict"],
            "literal_excerpt": row.get("literal_excerpt", ""),
            "excerpt_char_start": row.get("excerpt_char_start"),
            "excerpt_char_end": row.get("excerpt_char_end"),
            "literal_excerpt_sha256": row.get("literal_excerpt_sha256"),
        }
        for row in window_results
    ]
    notes = list(selected_notes or [])[:32]
    derivations = list(bounded_derivations or [])[:8]
    return (
        "FINAL FULL-CONTEXT AGGREGATION. The window ledger collectively covered every supplied-context "
        "character. Rebind the exact question and candidate response. A not_enough_information window means "
        "only that its local span was silent; it is never counter-evidence. If any aligned window refutes the "
        "response, do not let an unrelated support override it. Unresolved support/refutation across different "
        "entities, slots, event phases, dates, operations, or ambiguous coreference is not_enough_information. "
        "Use no outside knowledge.\n\n"
        f"QUESTION (literal):\n{question}\n\nCANDIDATE RESPONSE (literal):\n{response}\n\n"
        "PER-WINDOW STRUCTURED RESULTS AND LITERAL EXCERPTS:\n" + _canonical(compact)
        + "\n\nFULL-CONTEXT ROUTER SOURCE-LINKED NOTES (advisory; offsets/hashes bind them to supplied context):\n"
        + _canonical(notes)
        + "\n\nBOUNDED DERIVATION CANDIDATES (advisory; verify operator, operands, slot and unit):\n"
        + _canonical(derivations)
        + "\n\nFROZEN EXACT LEXICAL-SHELL POLICY (nonterminal; ordinary retrieval forbidden):\n"
        + _canonical(lexical_policy or {"lookup_mode": "not_applicable", "generic_retrieval_invoked": False})
    )


def build_contextual_policy_packet(
    context: str,
    question: str,
    response: str,
    router: Callable[..., dict[str, Any]],
    lexical_records: list[dict[str, Any]] | None = None,
    *,
    max_windows: int = 8,
    full_context_char_limit: int = 6000,
) -> tuple[str, dict[str, Any]]:
    """Build a context-only prompt plus restart-safe diagnostics."""
    if not all(isinstance(value, str) for value in (context, question, response)):
        raise TypeError("context, question and response must be strings")
    if not context.strip() or not question.strip():
        raise ValueError("context and question must be non-empty")

    route = router(context, question, response, max_windows=max_windows)
    if route.get("external_retrieval_allowed") is not False:
        raise ValueError("context router must explicitly forbid external retrieval")
    if route.get("context_sha256") != _sha(context):
        raise ValueError("context router hash mismatch")

    window_calls: list[dict[str, Any]] = []
    if len(context) <= full_context_char_limit:
        evidence = context
        evidence_mode = "complete_literal_supplied_context"
        full_context_inference_coverage = True
    else:
        coverage = full_coverage_windows(context)
        excerpt_chars = max(48, min(320, 3200 // len(coverage)))
        window_calls = [
            {
                **{key: window[key] for key in (
                    "window_index", "context_char_start", "context_char_end", "literal_span_sha256"
                )},
                **_literal_question_excerpt(window, question, excerpt_chars),
                "user": build_window_adjudication_user(window, question, response, len(coverage)),
            }
            for window in coverage
        ]
        # The ordinary single-call payload is never used for a long context;
        # it is deliberately tiny so no truncated projection can be mistaken
        # for the evidence universe.
        evidence = "[FULL-COVERAGE WINDOW PASSES REQUIRED BEFORE AGGREGATION]"
        evidence_mode = "all_literal_windows_then_final_aggregation"
        full_context_inference_coverage = True

    signals = detected_policy_families(context, question, response)
    lexical_policy = contextual_exact_lexical_policy(question, response, lexical_records)
    aggregation_derivations = [
        value for value in list(route.get("bounded_derivation_candidates") or [])[:8]
        if isinstance(value, dict) and value.get("source_linked") is True
    ]
    operand_ids = {
        str(note_id)
        for value in aggregation_derivations
        for note_id in value.get("operand_note_ids", [])
    }
    routed_notes = list(route.get("selected_notes") or [])
    routed_notes.sort(key=lambda note: (str(note.get("note_id")) not in operand_ids, int(note.get("context_char_start", 0))))
    aggregation_notes = []
    note_budget = 2400
    used_note_chars = 0
    for note in routed_notes[:32]:
        start = int(note["context_char_start"])
        end = int(note["context_char_end"])
        literal = str(note["literal_text"])
        if context[start:end] != literal or _sha(literal) != note["literal_span_sha256"]:
            raise ValueError("router selected note does not replay against supplied context")
        serialized_chars = len(_canonical(note))
        if aggregation_notes and used_note_chars + serialized_chars > note_budget:
            continue
        aggregation_notes.append(note)
        used_note_chars += serialized_chars
    notes = {
        "selected_notes": aggregation_notes,
        "bounded_derivation_candidates": aggregation_derivations,
    }
    contract = {
        "version": VERSION,
        "evidence_universe": "only_the_literal_supplied_context",
        "external_retrieval_allowed": False,
        "outside_world_fact_rescue_allowed": False,
        "evidence_mode": evidence_mode,
        "full_context_inference_coverage": full_context_inference_coverage,
        "every_context_character_processed": True,
        "detected_policy_families": signals,
        "checks_in_order": [
            "validate question premise and whether the requested slot is answerable",
            "bind exact entity relation property answer type event phase time polarity comparator and unit",
            "seek direct support and direct contradiction for that exact slot",
            "apply only supplied definition theory rule formula and bounded operands",
            "separate refuted from missing ambiguous conflicting or locally silent window evidence",
            "compare Unicode only through bounded NFC digit joiner and attested word-break views; near-looking forms are not automatically equal",
        ],
        "verdict_rule": {
            "supported": "the complete response follows for the exact requested slot",
            "refuted": "the supplied context establishes an incompatible value or claim",
            "not_enough_information": "required evidence is absent omitted ambiguous conflicting or premise-invalid",
        },
        "fixed_lexical_exception": lexical_policy,
    }
    user = (
        "CONTEXT-ONLY VERIFICATION CONTRACT:\n" + _canonical(contract)
        + "\n\nQUESTION (literal):\n" + question
        + "\n\nSUPPLIED CONTEXT EVIDENCE (literal; this is the only evidence universe):\n" + evidence
        + "\n\nSOURCE-LINKED EXTRACTIVE NOTES (advisory, never new evidence):\n" + _canonical(notes)
        + "\n\nCANDIDATE RESPONSE (verify every material claim):\n" + response
    )
    diagnostic = {
        **route,
        "context_policy_version": VERSION,
        "policy_family_inventory_count": len(CANONICAL_POLICY_FAMILIES),
        "canonical_policy_family_count": len(CANONICAL_POLICY_FAMILIES),
        "engineered_evaluation_cell_count": len(ENGINEERED_EVALUATION_CELLS),
        "operation_axis_count": len(OPERATION_AXIS),
        "canonical_policy_families": list(CANONICAL_POLICY_FAMILIES),
        "engineered_evaluation_cells": list(ENGINEERED_EVALUATION_CELLS),
        "operation_axis": list(OPERATION_AXIS),
        "detected_policy_families": signals,
        "evidence_mode": evidence_mode,
        "full_context_inference_coverage": full_context_inference_coverage,
        "context_literal": unicode_receipt(context),
        "question_literal": unicode_receipt(question),
        "response_literal": unicode_receipt(response),
        "prompt_sha256": _sha(user),
        "requires_window_aggregation": bool(window_calls),
        "window_calls": window_calls,
        "window_count": len(window_calls) if window_calls else 1,
        "full_context_character_count": len(context),
        "aggregation_selected_notes": aggregation_notes,
        "aggregation_bounded_derivations": aggregation_derivations,
        "contextual_lexical_policy": lexical_policy,
    }
    diagnostic.pop("receipt_sha256", None)
    diagnostic["receipt_sha256"] = _sha(_canonical(diagnostic))
    return user, diagnostic


__all__ = [
    "VERSION", "CANONICAL_POLICY_FAMILIES", "ENGINEERED_EVALUATION_CELLS",
    "OPERATION_AXIS", "comparison_view", "unicode_receipt",
    "detected_policy_families", "full_coverage_windows",
    "build_window_adjudication_user", "build_aggregation_user",
    "build_contextual_policy_packet",
]


In [ ]:
from pipeline.phase2_sparse_retrieval import build as build_sparse_retrieval
from pipeline.phase2_mmap_retrieval import load_mmap_index, close_mmap_index
from pipeline.phase2_composite_fts_retrieval import load as load_fts, retrieve as retrieve_fts
from pipeline.contextual_note_taker_fallback import rank_context_windows

def normalized_text(value: object) -> str:
    return str(value or "").replace("\r\n", "\n").replace("\r", "\n").strip()

rows = []
for source_index, raw in test.iterrows():
    context = normalized_text(raw.context)
    rows.append({
        "example_id": str(raw.id), "source_index": int(source_index),
        "model_prompt_bn": normalized_text(raw.prompt_bn),
        "model_response_bn": normalized_text(raw.response_bn),
        "model_context_bn": context, "context_available": has_context(context),
        "formatting_provenance": {"status": "newline_normalization_only"},
    })

def typed_grammar_operation(prompt: str) -> str:
    folded=" ".join(prompt.casefold().split())
    families=(
        ("antonym",r"বিপরীত(?:ার্থক)?\s*শব্দ"),("sandhi",r"সন্ধি"),("samas",r"সমাস"),
        ("idiom",r"বাগধারা|প্রবাদ"),("prefix_suffix",r"উপসর্গ|প্রত্য[য়য়]"),
        ("one_word_expression",r"এক\s*কথা[য়য়]\s*প্রকাশ"),("word_pair_register",r"শব্দজো[ড়ড়]া|গুরুচণ্ডালী|তৎসম|তদ্ভব|রেজিস্টার"),
        ("natva_satva",r"ণত্ব|নত্ব|ষত্ব"),("spelling_grammar_rule",r"শুদ্ধ\s*বানান|ব্যাকরণ(?:ের)?\s*নিয়ম|ব্যাকরণ(?:ের)?\s*নিয়ম"),
        ("definition_theory_rule",r"সংজ্ঞা|তত্ত্ব|সূত্র|বিধান|কাকে\s*বলে|কি\s*বোঝা[য়য়]|কী\s*বোঝা[য়য়]"),
    )
    return next((name for name,pattern in families if re.search(pattern,folded)),"")

closed = [row for row in rows if not row["context_available"]]
typed_context=[]
for row in rows:
    operation=typed_grammar_operation(row["model_prompt_bn"])
    if row["context_available"] and operation:
        proxy=dict(row); proxy["example_id"]="ctxgrammar:"+row["example_id"]
        proxy["context_available"]=False; proxy["typed_context_grammar_operation"]=operation
        typed_context.append(proxy)
retrieval_queries=closed+typed_context
closed_input = WORK_DIR / "morichika_closed_inputs.jsonl"
closed_input.write_text("".join(canonical_json(r)+"\n" for r in retrieval_queries), encoding="utf-8")

# Mandatory source-family health gate: every admitted family is hash-bound,
# openable, and independently search-probed before competition retrieval.
counts = json.loads((MORICHIKA_ROOT / "SOURCE_COUNTS.json").read_text(encoding="utf-8-sig"))
expected_counts = counts["records_by_source"]
health = {"manifest_id": EXPECTED_MANIFEST_ID, "package_id": EXPECTED_PACKAGE_ID,
          "declared_files": len(MORICHIKA_MANIFEST["files"]), "sources": {}, "closed_rows": len(closed),
          "typed_context_exact_grammar_lookup_rows":len(typed_context),"ordinary_context_external_retrieval":False}

mmap_dirs = sorted((MORICHIKA_ROOT / "artifacts/retrieval").glob("*_mmap_v2"))
seen = set()
for directory in mmap_dirs:
    manifest = json.loads((directory / "manifest.json").read_text(encoding="utf-8-sig"))
    sid = str(manifest.get("source_id", ""))
    if sid not in expected_counts or sid in seen: continue
    index = load_mmap_index(directory)
    try:
        probe = None
        for probe_index in range(min(512, len(index["records"]))):
            candidate_probe=index["records"][probe_index]
            if str(candidate_probe.get("question") or "").strip() and (
                candidate_probe.get("answers") or candidate_probe.get("choices") or candidate_probe.get("records")
                or str(candidate_probe.get("supporting_text") or "").strip()
            ):
                probe=candidate_probe; break
        if probe is None:
            raise RuntimeError(f"mmap known-answer record unavailable: {sid}")
        question = str(probe.get("question") or "")
        key = str(probe.get("normalized_question") or "")
        exact = index["exact_lookup"].get(key, [])
        query = index["vectorizer"].transform([question])
        scores = query @ index["matrix"].T
        top_score = float(scores.max()) if scores.nnz else 0.0
        if not question or top_score <= 0:
            raise RuntimeError(f"mmap known-answer probe failed: {sid}")
        health["sources"][sid] = {"kind":"mmap", "path":str(directory), "records":int(manifest["records"]),
                                  "probe_question":question[:120],"probe_answer_count":len(probe.get("answers") or probe.get("records") or []),
                                  "probe_exact_hits":len(exact), "probe_top_score":top_score}
        seen.add(sid)
    finally:
        close_mmap_index(index)
        del index
        gc.collect()

fts_dir = MORICHIKA_ROOT / "artifacts/retrieval/phase2_strict_rights_safe_fts_v3_final"
fts_manifest, connection = load_fts(fts_dir)
try:
    for sid in [str(x[0]) for x in connection.execute("SELECT DISTINCT source_id FROM evidence ORDER BY source_id")]:
        record = connection.execute("SELECT question,title,supporting_text FROM evidence WHERE source_id=? ORDER BY id LIMIT 1", (sid,)).fetchone()
        query = str(record[0] or record[1] or record[2] or "").strip()
        hits = retrieve_fts(connection, query, top_k=8, model_facing_only=False, source_ids=[sid])
        if not query or not any(str(hit.get("source_id")) == sid for hit in hits):
            raise RuntimeError(f"FTS known-answer probe failed: {sid}")
        health["sources"][sid] = {"kind":"fts", "path":str(fts_dir), "records":int(expected_counts[sid]),
                                  "probe_query":query[:120], "probe_hits":len(hits)}
        seen.add(sid)
finally:
    connection.close()

lexical_path = MORICHIKA_ROOT / "artifacts/retrieval/phase2_lexical_seed_v2/lexical_seed_records.jsonl"
lexical_records = [json.loads(line) for line in lexical_path.read_text(encoding="utf-8-sig").splitlines() if line.strip()]
lex_by_source = defaultdict(list)
for record in lexical_records: lex_by_source[str(record["source_id"])].append(record)
for sid, records_for_source in lex_by_source.items():
    probe = records_for_source[0]
    if not str(probe.get("term_key") or "") or not probe.get("accepted_answers"):
        raise RuntimeError(f"lexical known-answer probe failed: {sid}")
    declared_sid="lexical::"+sid
    health["sources"][declared_sid] = {"kind":"lexical_exact", "path":str(lexical_path), "records":len(records_for_source),
                              "probe_term":str(probe["term_key"]), "probe_answers":len(probe["accepted_answers"])}
    seen.add(declared_sid)

missing_sources = sorted(set(expected_counts) - seen)
if missing_sources:
    raise RuntimeError(f"MORICHIKA admitted source families inaccessible: {missing_sources}")
for sid, expected in expected_counts.items():
    if int(health["sources"][sid]["records"]) != int(expected):
        raise RuntimeError(f"source record-count mismatch: {sid}")

retrieval_dir = WORK_DIR / "morichika_closed_retrieval"
retrieval_receipt = build_sparse_retrieval(
    closed_input, retrieval_dir, top_k=8, batch_size=128,
    composite_cache_dir=fts_dir, composite_query_mode="all_closed",
)
retrieved_rows = [json.loads(line) for line in (retrieval_dir/"retrieval.jsonl").read_text(encoding="utf-8-sig").splitlines() if line.strip()]
retrieval_by_id = {str(r["example_id"]): r for r in retrieved_rows}
if len(retrieval_by_id) != len(retrieval_queries):
    raise RuntimeError("closed retrieval output cardinality mismatch")
candidate_rows = sum(bool(retrieval_by_id[r["example_id"]].get("retrieval_candidates")) for r in closed)
if closed and candidate_rows == 0:
    raise RuntimeError("all_closed retrieval returned zero candidates")
health["retrieval_receipt"] = retrieval_receipt
health["closed_rows_with_candidates"] = candidate_rows
health["closed_candidate_coverage"] = candidate_rows / max(1, len(closed))
(WORK_DIR/"morichika_retrieval_health.json").write_text(json.dumps(health, ensure_ascii=False, indent=2), encoding="utf-8")
print("MORICHIKA retrieval health:", len(health["sources"]), "families; closed hits", candidate_rows, "/", len(closed))

def exact_lexical(prompt: str, response: str, limit: int = 3) -> list[dict]:
    p = " ".join(prompt.casefold().split()); r = " ".join(response.casefold().split())
    operation = "antonym_lookup" if "বিপরীত" in p else "idiom_meaning_lookup" if ("বাগধারা" in p or "প্রবাদ" in p) else "prefix_origin_classification" if "উপসর্গ" in p else ""
    if not operation: return []
    prompt_tokens=set(re.findall(r"[A-Za-z0-9\u0980-\u09ff]+",p))
    generic={"অর্থ","মানে","শব্দ","শাব্দিক","বিপরীত","বিপরীতার্থক","বাগধারা","প্রবাদ","উপসর্গ","কি","কী","কোন","কোনটি"}
    found=[]
    for item in lexical_records:
        if item.get("operation") != operation or item.get("conflict_status") != "none": continue
        terms=[str(item.get("term_key") or "")]+[str(v) for v in item.get("display_terms", [])]
        # Generic shell words (for example only ``শাব্দিক অর্থ``) are not an
        # operand.  Require an exact distinctive record-key token in prompt.
        operand_tokens={token for term in terms for token in re.findall(r"[A-Za-z0-9\u0980-\u09ff]+"," ".join(term.casefold().split()))
                        if token not in generic and len(token)>=2}
        if not operand_tokens or not (operand_tokens & prompt_tokens): continue
        accepted=[" ".join(str(v).casefold().split()) for v in item.get("accepted_answers", [])]
        contrast=[" ".join(str(v).casefold().split()) for v in item.get("contrast_answers", [])]
        role="support_candidate" if r in accepted else "counter_candidate" if r in contrast else "neutral_candidate"
        found.append({"source_id":item.get("source_id"),"operation":operation,"term_key":item.get("term_key"),
                      "display_terms":item.get("display_terms",[]),"conflict_status":"none","sense":item.get("sense"),
                      "register":item.get("register"),"etymological_class":item.get("etymological_class"),
                      "accepted_answers":item.get("accepted_answers",[])[:8],"contrast_answers":item.get("contrast_answers",[])[:8],"evidence_role":role})
        if len(found)>=limit: break
    return found

def query_alignment_signature(prompt: str, response: str) -> dict:
    joined=" ".join(prompt.casefold().split())
    tokens=[t for t in re.findall(r"[A-Za-z0-9\u0980-\u09ff]+",joined) if len(t)>=2]
    numbers=re.findall(r"[+-]?[0-9০-৯]+(?:[.,][0-9০-৯]+)?",prompt)
    units=[u for u in ("সাল","বছর","দিন","মাস","কিলোমিটার","মিটার","টাকা","শতাংশ","জন","টি") if u in joined]
    negations=[n for n in ("না","নয়","নয়","নেই","ব্যতীত") if n in tokens]
    relation=[r for r in ("কবে","কোথায়","কোথায়","কে","কার","জন্ম","মৃত্যু","শপথ","অর্থ","বিপরীত","প্রতিষ্ঠাতা","স্রষ্টা","বয়স","বয়স") if r in joined]
    answer_type="date" if any(x in joined for x in ("কবে","তারিখ","কত সালে")) else "person" if re.search(r"(?:কে|কার)\??$",joined) else "location" if ("কোথায়" in joined or "কোথায়" in joined) else "number" if "কত" in joined else "lexical" if any(x in joined for x in ("অর্থ","বিপরীত","বাগধারা","উপসর্গ")) else "text"
    return {"substantive_query_tokens":tokens[:24],"relation_property_cues":relation,"numbers_dates":numbers,"units":units,
            "negation":negations,"answer_type":answer_type,"response_proposition":response[:240]}

def compact_closed_evidence(item: dict, lexical: list[dict], prompt: str, response: str, limit: int = 6) -> tuple[str,list[str],dict]:
    raw_candidates=list(item.get("retrieval_candidates") or [])
    def tier(value, default=99):
        try: return int(value)
        except (TypeError,ValueError): return default
    def alignment_key(candidate):
        # Exact requested relation/slot/answer type precedes authority.  This
        # prevents a wrong-relation passage from starving a lower-ranked exact
        # NCTB/date answer while retaining deterministic authority tie-breaks.
        return (
            candidate.get("model_facing_eligible") is not True,
            tier(candidate.get("semantic_alignment_tier")),
            tier(candidate.get("slot_compatibility_tier")),
            tier(candidate.get("policy_compatibility_tier")),
            candidate.get("number_set_match") is False,
            candidate.get("negation_set_match") is False,
            tier(candidate.get("authority_tier")),
            candidate.get("exact_normalized") is not True,
            str(candidate.get("evidence_role") or "neutral_candidate") == "neutral_candidate",
            tier(candidate.get("rank")),
            str(candidate.get("source_id") or ""),
        )
    signature=query_alignment_signature(prompt,response)
    stop={"কি","কী","কে","কার","কোন","কোনটি","কত","কবে","কোথায়","কোথায়","হয়","হয়","ছিল","আছে","একটি","এই","ও","এবং","সরকার"}
    relation_words=set(signature["relation_property_cues"]+[
        "গ্রহণ","গঠিত","প্রতিষ্ঠিত","অর্থ","বিপরীত","জন্ম","মৃত্যু","বয়স","বয়স","তারিখ","সাল"
    ])
    entity_tokens={t for t in signature["substantive_query_tokens"] if t not in stop and t not in relation_words and len(t)>=3}
    def hard_aligned(candidate):
        text=" ".join([str(candidate.get("question") or ""),str(candidate.get("supporting_text") or "")]).casefold()
        candidate_tokens=set(re.findall(r"[A-Za-z0-9\u0980-\u09ff]+",text))
        entity_ok=not entity_tokens or bool(entity_tokens & candidate_tokens)
        relation_cues={c for c in signature["relation_property_cues"] if c not in {"কবে","কে","কার","কোথায়","কোথায়"}}
        relation_ok=not relation_cues or bool(relation_cues & candidate_tokens)
        return (candidate.get("model_facing_eligible") is True and tier(candidate.get("semantic_alignment_tier"))<=1
                and tier(candidate.get("slot_compatibility_tier"))==0 and tier(candidate.get("policy_compatibility_tier"))<=1
                and candidate.get("number_set_match") is not False and candidate.get("negation_set_match") is not False
                and entity_ok and relation_ok)
    admitted=[candidate for candidate in raw_candidates if hard_aligned(candidate)]
    ranked=sorted(admitted,key=alignment_key)
    # Keep the best aligned candidate first, then preserve source diversity
    # before filling remaining prompt slots by the same alignment order.
    candidates=[]; used_sources=set()
    for candidate in ranked:
        sid=str(candidate.get("source_id") or "")
        if sid not in used_sources:
            candidates.append(candidate); used_sources.add(sid)
        if len(candidates)>=limit: break
    if len(candidates)<limit:
        for candidate in ranked:
            if candidate not in candidates: candidates.append(candidate)
            if len(candidates)>=limit: break
    lines=["QUERY_RESPONSE_ALIGNMENT "+canonical_json(signature)]+["EXACT_LEXICAL "+canonical_json(x) for x in lexical]
    source_ids=[]; roles=Counter()
    for candidate in candidates[:limit]:
        sid=str(candidate.get("source_id") or ""); source_ids.append(sid)
        role=str(candidate.get("evidence_role") or "neutral_candidate"); roles[role]+=1
        lines.append(canonical_json({k:candidate.get(k) for k in (
            "source_id","authority_class","authority_tier","evidence_role","question","answers","choices",
            "supporting_text","exact_normalized","semantic_alignment_tier","slot_compatibility_tier",
            "policy_compatibility_tier","number_set_match","negation_set_match","response_answer_relation")}))
    diagnostics={"query_alignment_signature":signature,"candidate_count":len(raw_candidates),"hard_aligned_candidate_count":len(admitted),
                 "misaligned_candidates_excluded":len(raw_candidates)-len(admitted),"prompt_candidate_count":len(candidates),"source_ids":sorted(set(source_ids)),"evidence_roles":dict(roles),
                 "merged_source_candidate":item.get("merged_source_candidate"),"retrieval_miss_means":"NEI_not_refutation"}
    return "\n".join(lines) if lines else "[NO ALIGNED SOURCE CANDIDATE: treat as NEI, never refutation]", sorted(set(source_ids)), diagnostics

test_aug=test.copy(); test_aug["row_key"]="test_"+test_aug.id.astype(str)
packets=[]
for row in rows:
    if row["context_available"]:
        lexical=exact_lexical(row["model_prompt_bn"],row["model_response_bn"])
        user_packet, route = build_contextual_policy_packet(row["model_context_bn"],row["model_prompt_bn"],row["model_response_bn"],rank_context_windows,
                                                             lexical_records=lexical,max_windows=8,full_context_char_limit=6000)
        operation=typed_grammar_operation(row["model_prompt_bn"])
        grammar_sources=[]; grammar_diag={"lookup_mode":"not_applicable","ordinary_context_external_retrieval":False}
        if operation:
            proxy_item=retrieval_by_id["ctxgrammar:"+row["example_id"]]
            exact_item=dict(proxy_item)
            exact_item["retrieval_candidates"]=[c for c in proxy_item.get("retrieval_candidates",[]) if c.get("exact_normalized") is True]
            grammar_evidence,grammar_sources,grammar_diag=compact_closed_evidence(exact_item,lexical,row["model_prompt_bn"],row["model_response_bn"],limit=4)
            grammar_diag.update({"lookup_mode":"typed_context_exact_grammar_exception","operation":operation,"fuzzy_candidates_exposed":False})
            user_packet += "\n\nTYPED GRAMMAR CLOSED-LOOKUP EXCEPTION (exact operation+operand/key+sense/register only; nonterminal):\n"+grammar_evidence
        packets.append({"morichika_lane":"contextual","phase2_rag_evidence":"","morichika_source_ids":grammar_sources,
                        "morichika_retrieval_diagnostics":grammar_diag,"morichika_context_packet":user_packet,
                        "morichika_context_receipt":route})
    else:
        item=retrieval_by_id[row["example_id"]]; lexical=exact_lexical(row["model_prompt_bn"],row["model_response_bn"])
        evidence,sources,diag=compact_closed_evidence(item,lexical,row["model_prompt_bn"],row["model_response_bn"])
        packets.append({"morichika_lane":"closed_book","phase2_rag_evidence":evidence,"morichika_source_ids":sources,
                        "morichika_retrieval_diagnostics":diag,"morichika_context_packet":"","morichika_context_receipt":{}})
packet_frame=pd.DataFrame(packets)
for col in packet_frame.columns: test_aug[col]=packet_frame[col].tolist()


In [ ]:
# MORICHIKA-owned, hash-verified offline llama.cpp runtime.
runtime_roots=[p.parent for p in INPUT_ROOT.rglob("runtime_manifest.json") if "morichika-offline-runtime" in str(p).lower()]
if len(runtime_roots)!=1: raise RuntimeError(f"expected one MORICHIKA runtime, got {runtime_roots}")
runtime_root=runtime_roots[0]
runtime_manifest=json.loads((runtime_root/"runtime_manifest.json").read_text(encoding="utf-8-sig"))
if runtime_manifest.get("dataset_id")!="ishtyy/morichika-offline-runtime-20260720": raise RuntimeError("wrong runtime dataset")
for spec in runtime_manifest["files"]:
    path=runtime_root/spec["path"]
    if not path.is_file() or path.stat().st_size!=int(spec["bytes"]) or file_sha256(path)!=spec["sha256"]:
        raise RuntimeError(f"runtime hash gate failed: {path}")
wheel=next(runtime_root.glob("llama_cpp_python-*.whl"))
subprocess.run([sys.executable,"-m","pip","install","--no-index","--no-deps","--force-reinstall",str(wheel)],check=True)
from llama_cpp import Llama, LlamaGrammar

def compact_field(value, limit=None):
    text=str(value or "").strip()
    return text if limit is None or len(text)<=limit else text[:limit]

def compact_context(value,max_chars=6000):
    text=str(value or "").strip()
    if len(text)<=max_chars: return text
    half=max_chars//2
    return text[:half]+"\n[...MIDDLE OMITTED ONLY AFTER SOURCE-LINKED FULL-COVERAGE NOTE EXTRACTION...]\n"+text[-half:]

def locate_model():
    candidates=[]
    for p in INPUT_ROOT.rglob("*.gguf"):
        name=p.name.casefold()
        if "31b" in name and ("q4_0" in name or "q4" in name): candidates.append(p)
    if len(candidates)!=1: raise RuntimeError(f"expected one official Gemma31B Q4 GGUF, got {candidates}")
    return candidates[0]

AB_GRAMMAR=LlamaGrammar.from_string('root ::= "A" | "B"')
def load_judge():
    model_path=locate_model()
    print("MORICHIKA model",model_path)
    return Llama(model_path=str(model_path),n_ctx=Q4_N_CTX,n_batch=Q4_N_BATCH,n_ubatch=Q4_N_UBATCH,
                 n_gpu_layers=-1,tensor_split=[0.5,0.5],main_gpu=0,flash_attn=True,
                 offload_kqv=True,logits_all=False,seed=SEED,verbose=False)

def cleanup_judge(judge):
    try: judge.close()
    except Exception: pass

def render_chat(system,user):
    return "<bos><start_of_turn>user\n"+system+"\n\n"+user+"<end_of_turn>\n<start_of_turn>model\n"

def one_letter(judge,prompt):
    # Reserve room for the verdict and truncate only from the left of the
    # rendered prompt; model-facing evidence is already ranked and bounded.
    tokens=judge.tokenize(prompt.encode("utf-8"),add_bos=False,special=True)
    if len(tokens)>=Q4_N_CTX-8:
        tokens=tokens[-(Q4_N_CTX-8):]
        prompt=judge.detokenize(tokens).decode("utf-8",errors="ignore")
    out=judge.create_completion(prompt=prompt,max_tokens=1,temperature=0.0,top_p=1.0,
                                repeat_penalty=1.0,grammar=AB_GRAMMAR,seed=SEED)
    letter=str(out["choices"][0]["text"]).strip().upper()[:1]
    if letter not in {"A","B"}: raise RuntimeError(f"constrained verdict failed: {out}")
    return letter

def score_rows(frame,cache_name,judge):
    cache=WORK_DIR/cache_name
    done={}
    if cache.is_file():
        prior=pd.read_csv(cache)
        done={str(r.row_key):r._asdict() for r in prior.itertuples(index=False)}
    records=[]
    for index,row in frame.iterrows():
        key=str(row.row_key); ph=_q4_prompt_hash(row)
        prior=done.get(key)
        if prior and str(prior.get("prompt_hash"))==ph:
            records.append(prior); continue
        normal=one_letter(judge,render_chat(SYSTEM_PROMPT,build_user_prompt(row,False)))
        reverse=one_letter(judge,render_chat(SYSTEM_PROMPT,build_user_prompt(row,True)))
        p_normal=1.0 if normal=="A" else 0.0
        p_reverse=1.0 if reverse=="B" else 0.0
        if p_normal==p_reverse:
            p_faithful=p_normal
        else:
            tie_user=build_user_prompt(row,False)+"\nThe two order-balanced passes disagreed. Re-check counterevidence, exact slot and lane boundary. Emit A or B."
            tie=one_letter(judge,render_chat(SYSTEM_PROMPT,tie_user))
            p_faithful=1.0 if tie=="A" else 0.0
        rec={"row_key":key,"prompt_hash":ph,"p_normal":p_normal,"p_reverse":p_reverse,
             "p_faithful":p_faithful,"order_gap":abs(p_normal-p_reverse),
             "normal_letter":normal,"reverse_letter":reverse}
        records.append(rec)
        if len(records)%CHECKPOINT_EVERY==0:
            pd.DataFrame(records).to_csv(cache,index=False)
            print("checkpoint",len(records),"/",len(frame))
    result=pd.DataFrame(records)
    result.to_csv(cache,index=False)
    return result


In [ ]:
SYSTEM_PROMPT = """You are MORICHIKA, a strict Bengali/English factuality verifier. Follow lane isolation and the frozen precedence exactly.
CONTEXTUAL lane: supplied context is the ordinary evidence boundary; never use factual/world retrieval. One narrow exception is recognized typed Bengali-language lexical or THEORY/RULE genres: antonym, idiom/bagdhara, prefix/suffix, sandhi, samas, one-word expression, word-pair/register/guruchandali, natva/satva, spelling and grammar rules may consult only exact canonical operation+operand/key+sense/register evidence. That lookup is nonterminal and must be reconciled with supplied context and response; fuzzy or generic factual rescue remains forbidden. Check premise/answerability first, then exact question slot and answer type, entity-relation-property, event phase, date/time/role/place, number/unit, negation/quantifier/comparator/modality/clause scope. Direct support is not mandatory: bounded calculation, age/duration/relative timeline, kinship, definition/formula/theory/rule application are allowed only from supplied operands. Unicode NFC/Bengali-vs-ASCII digits/joiner/attested OCR word break can be equivalent, but near-looking different words are not. Grammar requires the exact operation; antonym/idiom/prefix exact pair/sense/register and guruchandali matter; samas/sandhi/affix/natva/satva require exact rule and operands. Partial containment cannot hide an extra false claim. Missing/ambiguous/conflicting evidence is NOT_ENOUGH_INFORMATION, distinct from directly REFUTED.
CLOSED-BOOK lane: use MORICHIKA offline evidence as nonterminal evidence. Alignment precedes authority: exact operation, slot, entity, relation/property, answer type, date/number/unit, negation, cultural/as-of scope; then books/user OCR/BCS, curated datasets, then Wikipedia/other corroboration last. Consider support and counterevidence. A fuzzy hit is not proof; retrieval miss means NEI, never refutation. Lexical evidence is valid only for an exact recognized shell/key/sense/register with no conflict.
The frozen context inventory contains 17 canonical policy families, 26 engineered evaluation cells, and 15 operation axes. One material mismatch makes the response hallucinated. Do not reward fluency. Follow the A/B mapping exactly and emit one letter as the first token."""
MAIN_JUDGE_PROMPT_VERSION = "morichika-heavy-rag-gemma31b-v1-17x26x15"

def build_user_prompt(row: pd.Series, reverse: bool, context_max_chars: int=6000,
                      question_max_chars: int|None=None, response_max_chars: int|None=None) -> str:
    mapping="A=HALLUCINATED; B=FAITHFUL" if reverse else "A=FAITHFUL; B=HALLUCINATED"
    question=compact_field(row.prompt_bn,question_max_chars); response=compact_field(row.response_bn,response_max_chars)
    if row.morichika_lane == "contextual":
        packet=str(row.morichika_context_packet)
        # Context policy runtime already performs question-conditioned, source-linked window selection.
        if context_max_chars < 6000:
            packet=compact_context(packet,max_chars=max(400,context_max_chars))
        evidence="[EXTERNAL RETRIEVAL FORBIDDEN]"
    else:
        packet="[NO SUPPLIED CONTEXT]"
        evidence=compact_context(row.phase2_rag_evidence,max_chars=min(2400,max(800,context_max_chars)))
    lane_semantics=("CONTEXTUAL: faithful requires support/valid derivation from supplied context or the narrow exact typed-grammar exception. NEI differs from refutation diagnostically but is not faithful support."
                    if row.morichika_lane=="contextual" else
                    "CLOSED-BOOK: a retrieval miss/NEI is not evidence that the candidate is false; use model knowledge plus admitted nonterminal evidence and counterevidence.")
    return f"""<lane>{row.morichika_lane}</lane>
<lane_label_semantics>{lane_semantics}</lane_label_semantics>
<morichika_top_aligned_evidence_first>{evidence}</morichika_top_aligned_evidence_first>
<context_policy_packet>{packet}</context_policy_packet>
<exact_question>{question}</exact_question>
<candidate_response>{response}</candidate_response>
Decide whether the response is fully faithful under the lane-specific semantics. Mapping: {mapping}\nVerdict:"""

def _q4_prompt_hash(row: pd.Series) -> str:
    payload="\u241f".join([MAIN_JUDGE_PROMPT_VERSION,SYSTEM_PROMPT,str(row.morichika_lane),str(row.context),str(row.prompt_bn),str(row.response_bn),
                            str(row.phase2_rag_evidence),canonical_json(row.morichika_retrieval_diagnostics)])
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()[:20]


In [ ]:
test_cache_name="morichika_heavy_rag_gemma31b_v1_scores.csv"
judge=load_judge()
scores=score_rows(test_aug,test_cache_name,judge)
cleanup_judge(judge); del judge; gc.collect()
if len(scores)!=len(test_aug) or scores.row_key.astype(str).duplicated().any():
    raise RuntimeError("Gemma score cardinality failure; refusing fallback labels")
test_meta=test_aug.merge(scores[["row_key","prompt_hash","p_normal","p_reverse","p_faithful","order_gap","normal_letter","reverse_letter"]],on="row_key",how="left",validate="one_to_one")
if test_meta[["p_normal","p_reverse","p_faithful"]].isna().any().any():
    raise RuntimeError("missing model score; refusing silent default")
test_meta["final_label"]=(test_meta.p_faithful>=0.5).astype(int)
submission=test_meta[["id"]].copy(); submission["label"]=test_meta.final_label.astype(int)
if submission.id.tolist()!=sample_submission.id.tolist() or not submission.label.isin([0,1]).all():
    raise RuntimeError("submission contract failed")
submission.to_csv(WORK_DIR/"submission.csv",index=False)

diagnostics=test_meta[["id","morichika_lane","context","prompt_bn","response_bn","phase2_rag_evidence","morichika_source_ids",
                       "morichika_retrieval_diagnostics","morichika_context_receipt","p_normal","p_reverse","p_faithful","order_gap","final_label"]].copy()
for col in ("morichika_source_ids","morichika_retrieval_diagnostics","morichika_context_receipt"):
    diagnostics[col]=diagnostics[col].map(lambda x: canonical_json(x))
diagnostics.to_csv(WORK_DIR/"morichika_per_row_evidence_diagnostics.csv",index=False)
report={"version":"morichika-heavy-rag-gemma31b-v1","test_rows":len(test_meta),"context_rows":int((test_meta.morichika_lane=="contextual").sum()),
        "closed_rows":int((test_meta.morichika_lane=="closed_book").sum()),"manifest_id":EXPECTED_MANIFEST_ID,"package_id":EXPECTED_PACKAGE_ID,
        "source_family_count":len(health["sources"]),"closed_rows_with_candidates":candidate_rows,"model":Q4_MODEL_ID,
        "dual_order":True,"threshold":0.5,"fallback_labels_used":0,"submission_sha256":file_sha256(WORK_DIR/"submission.csv")}
(WORK_DIR/"morichika_run_receipt.json").write_text(json.dumps(report,ensure_ascii=False,indent=2),encoding="utf-8")
print(json.dumps(report,ensure_ascii=False,indent=2)); print(submission.label.value_counts().sort_index())
